# 1 - Imports

In [1]:
%reload_ext autoreload
%autoreload 2

In [2]:
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parents[0]))

In [3]:
import pandas as pd
import numpy as np

In [4]:
from src.utils import config, io, countries

/Users/hippolytegrandet/Desktop/Dev/country_risk_rating/.venv/lib/python3.9/site-packages/pypdf/_crypt_providers/_cryptography.py:32: CryptographyDeprecationWarning: ARC4 has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.ARC4 and will be removed from cryptography.hazmat.primitives.ciphers.algorithms in 48.0.0.
  from cryptography.hazmat.primitives.ciphers.algorithms import AES, ARC4


In [5]:
import wbgapi as wb
import pandas as pd
from typing import Dict, List

/Users/hippolytegrandet/Desktop/Dev/country_risk_rating/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


# 1 - Build Catalogs

In [7]:
wb_datasets = wb.source.info()
wb_datasets

id,name,code,concepts,lastupdated
1,Doing Business,DBS,3,2021-08-18
2,World Development Indicators,WDI,3,2026-01-28
3,Worldwide Governance Indicators,WGI,3,2024-11-05
5,Subnational Malnutrition Database,SNM,3,2016-03-21
6,International Debt Statistics,IDS,4,2025-12-03
11,Africa Development Indicators,ADI,3,2013-02-22
12,Education Statistics,EDS,3,2024-06-25
13,Enterprise Surveys,ESY,3,2022-03-25
14,Gender Statistics,GDS,3,2025-11-11
15,Global Economic Monitor,GEM,3,2025-12-17


In [ ]:
# wb_series = wb.series.info(db=6)
# wb.series.info(q='surplus')
wb.db = 2

In [53]:
WORLD_BANK_INDICATORS = {
    # Economic Growth & Output
    'NY.GDP.MKTP.CD': 'GDP (current US$)',
    'NY.GDP.MKTP.KD.ZG': 'GDP growth (annual %)',
    'NY.GDP.MKTP.PP.CD': 'GDP, PPP (current international $)',
    'NY.GDP.PCAP.CD': 'GDP per capita (current US$)',
    'NY.GDP.PCAP.KD.ZG': 'GDP per capita growth (annual %)',
    'NE.GDI.TOTL.KD.ZG': 'Gross capital formation (annual % growth)',
    'NE.GDI.TOTL.ZS': 'Gross capital formation (% of GDP)',
    'NV.IND.TOTL.CD': 'Industry (including construction), value added (current US$)',
    'NV.IND.TOTL.KD.ZG': 'Industry (including construction), value added (annual % growth)',
    'NV.IND.TOTL.ZS': 'Industry (including construction), value added (% of GDP)',
    'SL.IND.EMPL.ZS': 'Employment in industry (% of total employment) (modeled ILO estimate)',
    'NV.SRV.TOTL.CD': 'Services, value added (current US$)',
    'NV.SRV.TOTL.KD.ZG': 'Services, value added (annual % growth)',
    'NV.SRV.TOTL.ZS': 'Services, value added (% of GDP)',
    'NY.GNP.MKTP.KD.ZG': 'GNI growth (annual %)',
    'NY.GNP.MKTP.PP.CD': 'GNI, PPP (current international $)',
    'BG.GSR.NFSV.GD.ZS': 'Trade in services (% of GDP)',
    'CM.MKT.LCAP.GD.ZS': 'Market capitalization of listed domestic companies (% of GDP)',
    'CM.MKT.TRAD.GD.ZS': 'Stocks traded, total value (% of GDP)',
    'NY.GDS.TOTL.ZS': 'Gross domestic savings (% of GDP)',
    'NY.GNS.ICTR.ZS': 'Gross savings (% of GDP)',
    'NE.TRD.GNFS.ZS': 'Trade (% of GDP)',
    # Fiscal Position & Government Finance
    'GC.DOD.TOTL.GD.ZS': 'Central government debt, total (% of GDP)',
    'DT.DOD.DIMF.CD': 'Use of IMF credit (DOD, current US$)',
    # 'GC.BAL.CASH.GD.ZS': 'Cash surplus/deficit (% of GDP)',
    'GC.REV.XGRT.GD.ZS': 'Revenue, excluding grants (% of GDP)',
    'GC.XPN.TOTL.GD.ZS': 'Expense (% of GDP)',
    # Monetary Stability
    'FP.CPI.TOTL.ZG': 'Inflation, consumer prices (annual %)',
    'NY.GDP.DEFL.KD.ZG': 'Inflation, GDP deflator (annual %)',
    'FR.INR.RINR': 'Real interest rate (%)',
    'FM.LBL.BMNY.GD.ZS': 'Broad money (% of GDP)',
    'FM.LBL.BMNY.ZG': 'Broad money growth (annual %)',
    # External Sector & BOP
    'BN.CAB.XOKA.CD': 'Current account balance (current US$)',
    'NE.IMP.GNFS.ZS': 'Imports of goods and services (% of GDP)',
    'NE.EXP.GNFS.ZS': 'Exports of goods and services (% of GDP)',
    'BN.CAB.XOKA.GD.ZS': 'Current account balance (% of GDP)',
    'GC.TAX.INTT.RV.ZS': 'Taxes on international trade (% of revenue)',
    'BM.KLT.DINV.WD.GD.ZS': 'Foreign direct investment, net outflows (% of GDP)',
    'BX.KLT.DINV.WD.GD.ZS': 'Foreign direct investment, net inflows (% of GDP)',
    'DT.DOD.PVLX.GN.ZS': 'Present value of external debt (% of GNI)',
    'DT.DOD.DECT.GN.ZS': 'External debt stocks (% of GNI)', 
    'DT.DOD.DLXF.CD': 'External debt stocks, long-term (DOD, current US$)',
    'DT.DOD.DSTC.CD': 'External debt stocks, short-term (DOD, current US$)',
    'DT.TDS.DECT.CD': 'Debt service on external debt, total (TDS, current US$)',
    'NE.RSB.GNFS.ZS': 'External balance on goods and services (% of GDP)',
    # 'DT.DOD.DECT.PC.CD': 'Total external debt per capita (US$)',
    # 'FI.RES.TOTL.DT.ZS': 'Total reserves (% of total external debt)', # Slow
    'FI.RES.TOTL.MO': 'Total reserves in months of imports',
    'NE.EXP.GNFS.CD': 'Exports of goods and services (current US$)',
    'NE.IMP.GNFS.CD': 'Imports of goods and services (current US$)',
    'TX.VAL.TECH.CD': 'High-technology exports (current US$)',
    # Debt & Credit Sustainability
    # 'DT.TDS.DECT.EX.ZS': 'Total debt service (% of exports of goods, services and primary income)', # Slow
    'DT.TDS.DECT.GN.ZS': 'Total debt service (% of GNI)',
    'DT.DOD.DSTC.IR.ZS': 'Short-term debt (% of total reserves)',
    'GC.XPN.INTP.RV.ZS': 'Interest payments (% of revenue)',
    'GC.XPN.INTP.ZS': 'Interest payments (% of expense)',
    'FR.INR.DPST': 'Deposit interest rate (%)',
    'FR.INR.LEND': 'Lending interest rate (%)',
    'NE.CON.GOVT.ZS': 'General government final consumption expenditure (% of GDP)',
    'NE.CON.PRVT.ZS': 'Households and NPISHs final consumption expenditure (% of GDP)',
    'NE.CON.TOTL.ZS': 'Final consumption expenditure (% of GDP)',
    'NE.DAB.TOTL.ZS': 'Gross national expenditure (% of GDP)',
    # 'DT.INT.DECT.GN.ZS': 'Interest payments on external debt (% of GNI)',
    # 'DT.INR.DPPG': 'Average interest on new external debt commitments (%)',
    'FS.AST.DOMO.GD.ZS': 'Bank assets to GDP (%)',
    # 'FM.AST.CGOV.GD.ZS': 'Claims on central government (% of GDP)',
    # Labor & Demographics
    'SL.UEM.1524.ZS': 'Unemployment, youth total (% of total labor force ages 15-24) (modeled ILO estimate)',
    'SL.UEM.INTM.ZS': 'Unemployment with intermediate education (% of total labor force with intermediate education)',
    'SL.UEM.BASC.ZS': 'Unemployment with basic education (% of total labor force with basic education)',
    'SL.UEM.TOTL.ZS': 'Unemployment, total (% of total labor force) (modeled ILO estimate)',
    'EN.URB.LCTY.UR.ZS': 'Population in the largest city (% of urban population)',
    'SP.POP.TOTL': 'Population, total',
    'SP.POP.GROW': 'Population growth (annual %)',
    'SP.URB.TOTL.IN.ZS': 'Urban population (% of total population)',
    'EN.POP.DNST': 'Population density (people per sq. km of land area)',
    'SE.PRM.CUAT.ZS': 'Educational attainment, at least completed primary, population 25+ years, total (%) (cumulative)',
    'SP.POP.DPND': 'Age dependency ratio (% of working-age population)',
    'SL.TLF.CACT.ZS': 'Labor force participation rate, total (% of total population ages 15+) (modeled ILO estimate)',
    # 'SL.UEM.LTRM.ZS': 'Unemployment, long-term (% of total unemployment)',
    # Institutional Structure
    'GE.EST': 'Government Effectiveness: Estimate',
    'RQ.EST': 'Regulatory Quality: Estimate',
    'IC.BRE.BI.OS': 'B-READY: Business Insolvency: Overall Score',
    'IC.BRE.BE.OS': 'B-READY: Business Entry: Overall Score',
    'IQ.CPA.PADM.XQ': 'CPIA quality of public administration rating (1=low to 6=high)',
    'FS.AST.PRVT.GD.ZS': 'Domestic credit to private sector (% of GDP)',
    'FM.AST.PRVT.GD.ZS': 'Monetary Sector credit to private sector (% GDP)'
}

In [54]:
len(WORLD_BANK_INDICATORS)

78

In [56]:
df = wb.data.DataFrame(
    list(WORLD_BANK_INDICATORS.keys()),
    economy=OECD_COUNTRIES,
    time=range(2000,2025)
)
df

YR2000        YR2001        YR2002  \
economy series                                                           
CAN     BG.GSR.NFSV.GD.ZS     1.116449e+01  1.097318e+01  1.109022e+01   
        BM.KLT.DINV.WD.GD.ZS  6.263692e+00  4.984137e+00  3.843513e+00   
        BN.CAB.XOKA.CD        1.849442e+10  1.575782e+10  1.251291e+10   
        BN.CAB.XOKA.GD.ZS     2.483228e+00  2.132370e+00  1.645030e+00   
        BX.KLT.DINV.WD.GD.ZS  9.171024e+00  3.841921e+00  3.219148e+00   
...                                    ...           ...           ...   
USA     SP.POP.DPND           5.003583e+01  4.963850e+01  4.925237e+01   
        SP.POP.GROW           1.112769e+00  9.897414e-01  9.277975e-01   
        SP.POP.TOTL           2.821624e+08  2.849690e+08  2.876252e+08   
        SP.URB.TOTL.IN.ZS     7.907409e+01  7.930862e+01  7.954610e+01   
        TX.VAL.TECH.CD                 NaN           NaN           NaN   

                                    YR2003        YR2004        YR2005  \
economy series                                                           
CAN     BG.GSR.NFSV.GD.ZS     1.068568e+01  1.079402e+01  1.069667e+01   
        BM.KLT.DINV.WD.GD.ZS  2.616225e+00  4.365776e+00  2.325413e+00   
        BN.CAB.XOKA.CD        1.043294e+10  2.323555e+10  2.193310e+10   
        BN.CAB.XOKA.GD.ZS     1.164988e+00  2.263151e+00  1.869656e+00   
        BX.KLT.DINV.WD.GD.ZS  7.829676e-01  1.414659e-01  2.178164e+00   
...                                    ...           ...           ...   
USA     SP.POP.DPND           4.887488e+01  4.849792e+01  4.809568e+01   
        SP.POP.GROW           8.594817e-01  9.254840e-01  9.217132e-01   
        SP.POP.TOTL           2.901079e+08  2.928053e+08  2.955166e+08   
        SP.URB.TOTL.IN.ZS     7.977734e+01  7.999609e+01  8.019616e+01   
        TX.VAL.TECH.CD                 NaN           NaN           NaN   

                                    YR2006        YR2007        YR2008  \
economy series                                                           
CAN     BG.GSR.NFSV.GD.ZS     1.056020e+01  1.046603e+01  1.065149e+01   
        BM.KLT.DINV.WD.GD.ZS  3.816692e+00  4.434938e+00  5.710829e+00   
        BN.CAB.XOKA.CD        1.799024e+10  1.105081e+10  3.177945e+09   
        BN.CAB.XOKA.GD.ZS     1.363656e+00  7.523595e-01  2.046340e-01   
        BX.KLT.DINV.WD.GD.ZS  4.874101e+00  8.200559e+00  4.515137e+00   
...                                    ...           ...           ...   
USA     SP.POP.DPND           4.780328e+01  4.777358e+01  4.791874e+01   
        SP.POP.GROW           9.642539e-01  9.510552e-01  9.458653e-01   
        SP.POP.TOTL           2.983799e+08  3.012312e+08  3.040940e+08   
        SP.URB.TOTL.IN.ZS     8.037131e+01  8.051532e+01  8.062197e+01   
        TX.VAL.TECH.CD                 NaN  2.406635e+11  2.431320e+11   

                                    YR2009  ...        YR2015        YR2016  \
economy series                              ...                               
CAN     BG.GSR.NFSV.GD.ZS     1.108333e+01  ...  1.220573e+01  1.256738e+01   
        BM.KLT.DINV.WD.GD.ZS  2.745409e+00  ...  5.391157e+00  4.424490e+00   
        BN.CAB.XOKA.CD       -4.077526e+10  ... -5.469552e+10 -4.726258e+10   
        BN.CAB.XOKA.GD.ZS    -2.966282e+00  ... -3.513987e+00 -3.093111e+00   
        BX.KLT.DINV.WD.GD.ZS  1.524131e+00  ...  3.853895e+00  2.238350e+00   
...                                    ...  ...           ...           ...   
USA     SP.POP.DPND           4.809797e+01  ...  5.038028e+01  5.078853e+01   
        SP.POP.GROW           8.766513e-01  ...  7.979047e-01  7.856255e-01   
        SP.POP.TOTL           3.067715e+08  ...  3.218151e+08  3.243533e+08   
        SP.URB.TOTL.IN.ZS     8.068504e+01  ...  8.057080e+01  8.049766e+01   
        TX.VAL.TECH.CD        1.509781e+11  ...  1.753217e+11  1.739834e+11   

                                    YR2017        YR2018        YR2019  \
economy series                               

In [62]:
df = df.rename(columns={c: int(c[2:]) for c in df.columns})
df

2000          2001          2002  \
economy series                                                           
CAN     BG.GSR.NFSV.GD.ZS     1.116449e+01  1.097318e+01  1.109022e+01   
        BM.KLT.DINV.WD.GD.ZS  6.263692e+00  4.984137e+00  3.843513e+00   
        BN.CAB.XOKA.CD        1.849442e+10  1.575782e+10  1.251291e+10   
        BN.CAB.XOKA.GD.ZS     2.483228e+00  2.132370e+00  1.645030e+00   
        BX.KLT.DINV.WD.GD.ZS  9.171024e+00  3.841921e+00  3.219148e+00   
...                                    ...           ...           ...   
USA     SP.POP.DPND           5.003583e+01  4.963850e+01  4.925237e+01   
        SP.POP.GROW           1.112769e+00  9.897414e-01  9.277975e-01   
        SP.POP.TOTL           2.821624e+08  2.849690e+08  2.876252e+08   
        SP.URB.TOTL.IN.ZS     7.907409e+01  7.930862e+01  7.954610e+01   
        TX.VAL.TECH.CD                 NaN           NaN           NaN   

                                      2003          2004          2005  \
economy series                                                           
CAN     BG.GSR.NFSV.GD.ZS     1.068568e+01  1.079402e+01  1.069667e+01   
        BM.KLT.DINV.WD.GD.ZS  2.616225e+00  4.365776e+00  2.325413e+00   
        BN.CAB.XOKA.CD        1.043294e+10  2.323555e+10  2.193310e+10   
        BN.CAB.XOKA.GD.ZS     1.164988e+00  2.263151e+00  1.869656e+00   
        BX.KLT.DINV.WD.GD.ZS  7.829676e-01  1.414659e-01  2.178164e+00   
...                                    ...           ...           ...   
USA     SP.POP.DPND           4.887488e+01  4.849792e+01  4.809568e+01   
        SP.POP.GROW           8.594817e-01  9.254840e-01  9.217132e-01   
        SP.POP.TOTL           2.901079e+08  2.928053e+08  2.955166e+08   
        SP.URB.TOTL.IN.ZS     7.977734e+01  7.999609e+01  8.019616e+01   
        TX.VAL.TECH.CD                 NaN           NaN           NaN   

                                      2006          2007          2008  \
economy series                                                           
CAN     BG.GSR.NFSV.GD.ZS     1.056020e+01  1.046603e+01  1.065149e+01   
        BM.KLT.DINV.WD.GD.ZS  3.816692e+00  4.434938e+00  5.710829e+00   
        BN.CAB.XOKA.CD        1.799024e+10  1.105081e+10  3.177945e+09   
        BN.CAB.XOKA.GD.ZS     1.363656e+00  7.523595e-01  2.046340e-01   
        BX.KLT.DINV.WD.GD.ZS  4.874101e+00  8.200559e+00  4.515137e+00   
...                                    ...           ...           ...   
USA     SP.POP.DPND           4.780328e+01  4.777358e+01  4.791874e+01   
        SP.POP.GROW           9.642539e-01  9.510552e-01  9.458653e-01   
        SP.POP.TOTL           2.983799e+08  3.012312e+08  3.040940e+08   
        SP.URB.TOTL.IN.ZS     8.037131e+01  8.051532e+01  8.062197e+01   
        TX.VAL.TECH.CD                 NaN  2.406635e+11  2.431320e+11   

                                      2009  ...          2015          2016  \
economy series                              ...                               
CAN     BG.GSR.NFSV.GD.ZS     1.108333e+01  ...  1.220573e+01  1.256738e+01   
        BM.KLT.DINV.WD.GD.ZS  2.745409e+00  ...  5.391157e+00  4.424490e+00   
        BN.CAB.XOKA.CD       -4.077526e+10  ... -5.469552e+10 -4.726258e+10   
        BN.CAB.XOKA.GD.ZS    -2.966282e+00  ... -3.513987e+00 -3.093111e+00   
        BX.KLT.DINV.WD.GD.ZS  1.524131e+00  ...  3.853895e+00  2.238350e+00   
...                                    ...  ...           ...           ...   
USA     SP.POP.DPND           4.809797e+01  ...  5.038028e+01  5.078853e+01   
        SP.POP.GROW           8.766513e-01  ...  7.979047e-01  7.856255e-01   
        SP.POP.TOTL           3.067715e+08  ...  3.218151e+08  3.243533e+08   
        SP.URB.TOTL.IN.ZS     8.068504e+01  ...  8.057080e+01  8.049766e+01   
        TX.VAL.TECH.CD        1.509781e+11  ...  1.753217e+11  1.739834e+11   

                                      2017          2018          2019  \
economy series                                 

In [73]:
df = df.stack().unstack(level='series')
df = df.reset_index().rename(columns={'economy': 'ISO3_COUNTRY_CODE', 'level_1': 'YEAR'})
df

series,ISO3_COUNTRY_CODE,YEAR,BG.GSR.NFSV.GD.ZS,BM.KLT.DINV.WD.GD.ZS,BN.CAB.XOKA.CD,BN.CAB.XOKA.GD.ZS,BX.KLT.DINV.WD.GD.ZS,CM.MKT.LCAP.GD.ZS,CM.MKT.TRAD.GD.ZS,EN.POP.DNST,...,SL.TLF.CACT.ZS,SL.UEM.1524.ZS,SL.UEM.BASC.ZS,SL.UEM.INTM.ZS,SL.UEM.TOTL.ZS,SP.POP.DPND,SP.POP.GROW,SP.POP.TOTL,SP.URB.TOTL.IN.ZS,TX.VAL.TECH.CD
0,CAN,2000,11.164486,6.263692,1.849442e+10,2.483228,9.171024,103.500142,84.386383,3.422611,...,65.291,12.669,12.524,7.013,6.829,46.505468,0.931282,30685730.0,78.629300,NaN
1,CAN,2001,10.973177,4.984137,1.575782e+10,2.132370,3.841921,83.258605,60.346634,3.459990,...,65.412,12.869,12.926,7.225,7.219,46.049099,1.086199,31020855.0,78.841476,NaN
2,CAN,2002,11.090222,3.843513,1.251291e+10,1.645030,3.219148,116.831457,58.957746,3.497728,...,66.386,13.618,13.983,7.831,7.665,45.616213,1.084793,31359199.0,79.099023,NaN
3,CAN,2003,10.685677,2.616225,1.043294e+10,1.164988,0.782968,101.640346,56.563406,3.529323,...,67.111,13.624,13.743,7.738,7.574,45.224976,0.899227,31642461.0,79.391381,NaN
4,CAN,2004,10.794023,4.365776,2.323555e+10,2.263151,0.141466,114.690673,68.812786,3.562376,...,67.057,13.443,13.488,7.442,7.185,44.823966,0.932187,31938807.0,79.692306,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,USA,2020,5.750816,1.340573,-5.935040e+11,-2.818094,0.650821,197.383319,195.088022,36.248223,...,61.556,14.891,11.724,10.457,8.055,52.589682,0.408428,331577720.0,79.997452,1.416121e+11
96,USA,2021,5.964028,1.467509,-8.586340e+11,-3.682741,2.049986,208.228052,196.757397,36.305293,...,61.501,9.704,8.056,7.076,5.349,52.951709,0.157317,332099760.0,80.007477,1.692818e+11
97,USA,2022,6.541366,1.518646,-9.931420e+11,-3.878726,1.628168,157.384175,173.076167,36.514921,...,61.794,8.105,5.507,4.960,3.650,53.377634,0.575745,334017321.0,80.032498,1.918761e+11
98,USA,2023,6.620474,1.286395,-9.280170e+11,-3.400305,1.326183,179.463180,136.534444,36.819806,...,62.080,7.953,5.591,4.888,3.638,53.908800,0.831493,336806231.0,80.071561,2.085144e+11


In [31]:
def fetch_world_bank_indicators(
    indicators: Dict[str, str],
    countries: List[str],
    start_year: int,
    end_year: int,
) -> pd.DataFrame:
    '''
    Fetches annual World Bank indicators and returns a tidy DataFrame
    indexed by country and year.

    Parameters
    ----------
    indicators : dict
        Mapping from World Bank indicator codes to readable names
    countries : list
        List of ISO country codes (e.g. ['USA', 'FRA', 'DEU'])
    start_year : int
    end_year : int

    Returns
    -------
    pd.DataFrame
        Columns: country, year, <indicator_1>, <indicator_2>, ...
    '''

    records = []

    for wb_code, name in indicators.items():
        print(wb_code, name)

        data = wb.data.fetch(
            wb_code,
            economy=countries,
            time=range(start_year, end_year + 1),
        )

        for obs in data:
            if obs['time'][:2] != 'YR':
                print(obs['time'])
            records.append({
                'ISO3_COUNTRY_CODE': obs['economy'],
                'YEAR': int(obs['time'][2:]),
                'INDICATOR_NAME': name,
                'INDICATOR_CODE': obs['series'],
                'VALUE': obs['value']
            })

    df = pd.DataFrame(records)

    # Pivot to wide format: one row per country-year
    df = (
        df.pivot_table(
            index=['ISO3_COUNTRY_CODE', 'YEAR'],
            columns='INDICATOR_CODE',
            values='VALUE',
            aggfunc='mean',
        )
        .reset_index()
        .sort_values(['ISO3_COUNTRY_CODE', 'YEAR'])
    )

    return df

In [32]:
OECD_COUNTRIES = [
    'USA', 'CAN', 'GBR', 'FRA'
]

df = fetch_world_bank_indicators(
    indicators= WORLD_BANK_INDICATORS,
    countries=OECD_COUNTRIES,
    start_year=2000,
    end_year=2002,
)
df

NY.GDP.MKTP.CD GDP (current US$)
NY.GDP.MKTP.KD.ZG GDP growth (annual %)
NY.GDP.MKTP.PP.CD GDP, PPP (current international $)
NY.GDP.PCAP.CD GDP per capita (current US$)
NY.GDP.PCAP.KD.ZG GDP per capita growth (annual %)
NE.GDI.TOTL.KD.ZG Gross capital formation (annual % growth)
NE.GDI.TOTL.ZS Gross capital formation (% of GDP)
NV.IND.TOTL.CD Industry (including construction), value added (current US$)
NV.IND.TOTL.KD.ZG Industry (including construction), value added (annual % growth)
NV.IND.TOTL.ZS Industry (including construction), value added (% of GDP)
SL.IND.EMPL.ZS Employment in industry (% of total employment) (modeled ILO estimate)
NV.SRV.TOTL.CD Services, value added (current US$)
NV.SRV.TOTL.KD.ZG Services, value added (annual % growth)
NV.SRV.TOTL.ZS Services, value added (% of GDP)
NY.GNP.MKTP.KD.ZG GNI growth (annual %)
NY.GNP.MKTP.PP.CD GNI, PPP (current international $)
GC.DOD.TOTL.GD.ZS Central government debt, total (% of GDP)
DT.DOD.DIMF.CD Use of IMF credit (DOD, cur

INDICATOR_CODE,ISO3_COUNTRY_CODE,YEAR,BM.KLT.DINV.WD.GD.ZS,BN.CAB.XOKA.CD,BN.CAB.XOKA.GD.ZS,BX.KLT.DINV.WD.GD.ZS,EN.POP.DNST,EN.URB.LCTY.UR.ZS,FI.RES.TOTL.MO,FM.AST.PRVT.GD.ZS,...,RQ.EST,SE.PRM.CUAT.ZS,SL.IND.EMPL.ZS,SL.TLF.CACT.ZS,SL.UEM.1524.ZS,SL.UEM.TOTL.ZS,SP.POP.DPND,SP.POP.GROW,SP.POP.TOTL,SP.URB.TOTL.IN.ZS
0,CAN,2000,6.263692,1.849442e+10,2.483228,9.171024,3.422611,19.094607,1.161299,73.987738,...,1.464475,88.401908,23.294178,65.291,12.669,6.829,46.505468,0.931282,30685730.0,78.629300
1,CAN,2001,4.984137,1.575782e+10,2.132370,3.841921,3.459990,19.190606,1.319514,121.352832,...,NaN,89.244323,23.062835,65.412,12.869,7.219,46.049099,1.086199,31020855.0,78.841476
2,CAN,2002,3.843513,1.251291e+10,1.645030,3.219148,3.497728,19.257136,1.436241,117.839456,...,1.533917,89.569204,23.229868,66.386,13.618,7.665,45.616213,1.084793,31359199.0,79.099023
3,FRA,2000,12.755430,1.612518e+10,1.184839,3.041170,113.032120,20.888682,1.747474,NaN,...,0.914937,100.000000,26.385040,55.303,20.484,10.218,53.691972,0.686336,60918661.0,76.514398
4,FRA,2001,6.375834,2.096539e+10,1.529899,3.658998,113.859128,20.779218,1.615059,77.388386,...,NaN,100.000000,26.167381,55.122,17.910,8.610,53.826354,0.728994,61364377.0,76.907441
5,FRA,2002,3.551423,1.765101e+10,1.182704,3.454136,114.689938,20.683812,1.622623,76.528439,...,0.975889,100.000000,25.515929,55.399,18.846,8.702,53.855081,0.727033,61812142.0,77.254387
6,GBR,2000,17.471111,-3.014265e+10,-1.803224,9.818769,243.427909,15.470646,0.935057,114.294249,...,1.805725,NaN,25.264487,61.338,11.874,5.558,53.431486,0.357301,58892514.0,79.824141
7,GBR,2001,4.674152,-2.913980e+10,-1.759468,3.386783,244.366854,15.485286,0.876404,119.336886,...,NaN,NaN,24.745277,60.966,10.191,4.696,52.882333,0.384976,59119673.0,79.983830
8,GBR,2002,6.752278,-3.523323e+10,-1.967747,5.013055,245.403542,15.487838,1.007930,124.533460,...,1.708124,NaN,24.020180,61.254,10.779,5.037,52.496430,0.423337,59370479.0,80.114606
9,USA,2000,1.818076,-4.019280e+11,-3.920886,3.405783,30.797301,7.983857,0.854272,48.974905,...,1.696121,99.525737,22.669681,66.146,9.278,3.992,50.035833,1.112769,282162411.0,79.074090


In [80]:
df['YEAR'].value_counts()

YEAR
1999    15
2000    15
2001    15
2002    15
2003    15
2004    15
2005    15
2006    15
2007    15
2008    15
2009    15
2010    15
2011    15
2012    15
2013    15
2014    15
2015    15
2016    15
2017    15
2018    15
2019    15
2020    15
2021    15
2022    15
2023    15
2024    15
Name: count, dtype: int64